In [0]:
-- ============================================================================
-- SQL QUERY DOCUMENTATION
-- ============================================================================
-- This query demonstrates key SQL concepts:
-- 1. Query Parameters (dynamic values via UI widgets)
-- 2. IDENTIFIER() function (dynamic column/table names)
-- 3. Common Table Expressions (CTEs for complex queries)
-- 4. SQL Query Snippets (reusable code templates)
-- ============================================================================
-- ----------------------------------------------------------------------------
-- SECTION 1: BASIC QUERIES
-- ----------------------------------------------------------------------------
-- Simple SELECT: Retrieve all customer data
select * from customer_enr;
-- ----------------------------------------------------------------------------
-- SECTION 2: QUERY PARAMETERS
-- ----------------------------------------------------------------------------
-- Query parameters allow users to input values at runtime without editing SQL.
-- They appear as widgets above the query editor.
--
-- SYNTAX: :parameter_name
--
-- Active parameters in this query:
--   • param_1     : Category filter (String)
--   • para_col1   : First column name (String) = 'product_id'
--   • para_col2   : Second column name (String) = 'category'
--   • top_n       : Number of top products (String) = '3'
--
-- BENEFITS:
--   ✓ No need to edit SQL code for different values
--   ✓ User-friendly interface for non-SQL users
--   ✓ Safe from SQL injection
--   ✓ Values can be changed and query re-run instantly
-- ----------------------------------------------------------------------------
-- Example 1: Filter products by category using a parameter
-- The :param_1 will be replaced with the value from the parameter widget
select * from product_enr where category = :param_1;
-- ----------------------------------------------------------------------------
-- SECTION 3: IDENTIFIER() FUNCTION
-- ----------------------------------------------------------------------------
-- IDENTIFIER() allows dynamic column, table, or schema names at runtime.
-- Use when you need to parameterize object names (not just values).
--
-- SYNTAX: IDENTIFIER(:parameter_name)
--
-- USE CASES:
--   • Dynamic column selection
--   • Dynamic table/schema references
--   • Dynamic catalog paths
--
-- DIFFERENCE FROM REGULAR PARAMETERS:
--   :param         → Used for VALUES (WHERE col = :param)
--   IDENTIFIER(:p) → Used for OBJECT NAMES (SELECT IDENTIFIER(:p))
-- ----------------------------------------------------------------------------
-- Example 2: Dynamic column selection
-- This query selects columns based on parameter values
-- para_col1='product_id' and para_col2='category'
-- Result: SELECT product_id, category FROM product_enr
select IDENTIFIER(:para_col1), IDENTIFIER(:para_col2) from product_enr;
-- ----------------------------------------------------------------------------
-- SECTION 4: STATIC QUERY (FOR COMPARISON)
-- ----------------------------------------------------------------------------
-- Hard-coded version of the parameterized query above
-- Compare with the parameterized version to see the flexibility gained
select * from product_enr where category = 'Toys';
-- ----------------------------------------------------------------------------
-- SECTION 5: COMMON TABLE EXPRESSIONS (CTEs)
-- ----------------------------------------------------------------------------
-- CTEs are temporary named result sets that exist only during query execution.
-- They make complex queries more readable and maintainable.
--
-- SYNTAX:
--   WITH cte_name AS (
--     SELECT ...
--   )
--   SELECT * FROM cte_name
--
-- BENEFITS:
--   ✓ Break complex logic into manageable steps
--   ✓ Improve query readability
--   ✓ Reuse intermediate results multiple times
--   ✓ Better than subqueries for complex operations
--
-- MULTIPLE CTEs:
--   WITH first_cte AS (...),
--        second_cte AS (...)  
--   SELECT * FROM second_cte
--
-- NOTE: Separate multiple CTEs with commas, not closing parentheses
-- ----------------------------------------------------------------------------
-- Example 3: Top N Products by Sales Amount (Using CTEs)
-- This query demonstrates a two-step CTE approach:
-- CTE 1: join_query
-- Purpose: Join sales and product tables, aggregate total sales by product
with join_query as (
  select 
    sum(s.total_amount) as total_amount,
    p.product_name
  from sales_enr s 
  left join product_enr p on s.product_id = p.product_id
  group by p.product_name
),
-- CTE 2: ranked_query  
-- Purpose: Rank products by total sales using DENSE_RANK()
-- DENSE_RANK() assigns ranks without gaps (1, 2, 3, ...)
-- ORDER BY total_amount DESC ranks highest sales first
ranked_query as (
  select 
    *,
    dense_rank() over (order by total_amount desc) as rank
  from join_query
)
-- Final SELECT: Filter to top N products
-- Uses :top_n parameter to control how many products to return
-- Default: top_n = 3 (top 3 products)
select * from ranked_query where rank <= :top_n;
-- ============================================================================
-- SECTION 6: SQL QUERY SNIPPETS
-- ============================================================================
-- SQL Query Snippets are reusable code templates that speed up query writing.
-- They provide pre-formatted code patterns for common SQL operations.
--
-- HOW TO ACCESS SNIPPETS:
--   METHOD 1: Keyboard Shortcut
--     • Mac: Cmd + /
--     • Windows/Linux: Ctrl + /
--     • Opens snippet picker overlay
--
--   METHOD 2: Start Typing
--     • Type a snippet prefix (e.g., 'select', 'cte', 'case')
--     • Autocomplete shows matching snippets
--     • Press Tab or Enter to insert
--
--   METHOD 3: Command Palette
--     • Mac: Cmd + Shift + P
--     • Windows/Linux: Ctrl + Shift + P
--     • Type 'snippet' and select 'Insert Snippet'
--
-- BUILT-IN SNIPPETS:
--   'select'    → Basic SELECT statement template
--   'cte'       → Common Table Expression (WITH clause)
--   'case'      → CASE WHEN statement
--   'window'    → Window function template
--   'join'      → JOIN clause templates
--   'aggregate' → Aggregation query pattern
--
-- CREATING CUSTOM SNIPPETS:
--   1. Navigate to Workspace → Snippets
--   2. Click 'Create' → 'Snippet'
--   3. Define snippet name, trigger, and code template
--   4. Save and use with keyboard shortcut or typing trigger
--
-- SNIPPET BEST PRACTICES:
--   ✓ Use snippets for repetitive query patterns
--   ✓ Create custom snippets for team-specific patterns
--   ✓ Include parameter placeholders in snippets (:param_name)
--   ✓ Use descriptive trigger words for easy discovery
--
-- EXAMPLE CUSTOM SNIPPETS:
--   Trigger: 'topn'
--   Code:
--     WITH ranked AS (
--       SELECT *, ROW_NUMBER() OVER (ORDER BY :sort_col DESC) AS rn
--       FROM :table_name
--     )
--     SELECT * FROM ranked WHERE rn <= :n
--
--   Trigger: 'datefilter'
--   Code:
--     WHERE date_column BETWEEN :start_date AND :end_date
--
--   Trigger: 'paramselect'
--   Code:
--     SELECT IDENTIFIER(:column_param) FROM :table_param
--     WHERE IDENTIFIER(:filter_column) = :filter_value
--
-- WHY USE SNIPPETS:
--   ✓ Write queries faster (reduce typing)
--   ✓ Avoid syntax errors (pre-tested templates)
--   ✓ Standardize team coding patterns
--   ✓ Include best practices in templates
--   ✓ Reduce cognitive load (don't memorize syntax)
--
-- SNIPPETS vs PARAMETERS:
--   Snippets   → Reusable CODE templates
--   Parameters → Reusable VALUE inputs
--   Use both together for maximum productivity!
-- ============================================================================
-- ============================================================================
-- QUERY EXECUTION TIPS
-- ============================================================================
-- 1. Change parameter values using the widgets above the query
-- 2. Click 'Run' to execute with new parameter values
-- 3. Use Ctrl/Cmd + Enter to run the current statement only
-- 4. Use Ctrl/Cmd + Shift + Enter to run all statements
--
-- EXPERIMENT WITH:
--   • Change param_1 to different categories: 'Electronics', 'Books', etc.
--   • Change top_n to see more/fewer products: 5, 10, etc.
--   • Change para_col1 and para_col2 to select different columns
--   • Try using Cmd+/ (Mac) or Ctrl+/ (Win) to insert snippets
--   • Create custom snippets for your frequently-used query patterns
-- ============================================================================